# Phase 4: PPO Training with Causal Diversity

Trains a Proximal Policy Optimization (PPO) agent that ranks candidate news items by balancing click-through rate (CTR) and causal diversity impact (CDI).

## Scope Lock
- Phase 4 only: RL training with PPO.
- Uses `src.rl_agent.train_ppo.train_ppo()` to train an SB3 PPO model.
- Reward combines click reward (`R_click`) with CDI bonus from Phase 3.

## Inputs
- `data/scm_train.parquet` — Phase 1 feature table
- `artifacts/cdi_cache.pkl` — Phase 3 precomputed causal diversity scores

## Outputs
- `artifacts/ppo_model_final.zip` — trained PPO policy
- `artifacts/checkpoints/` — intermediate checkpoints
- `artifacts/tb_logs/` — TensorBoard logs

In [ ]:
import warnings
import logging
import pickle
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

_cwd = Path.cwd()
_root = _cwd.parent if _cwd.name == "notebooks" else _cwd
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.rl_agent.train_ppo import train_ppo

warnings.filterwarnings('ignore')
logging.getLogger('src.rl_agent').setLevel(logging.INFO)

DATA = _root / "data"
ARTIFACTS = _root / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print(f"Project root: {_root}")
print(f"Artifacts: {ARTIFACTS}")

## 2. Load Data and CDI Cache
Load the Phase 1 feature table and the Phase 3 CDI cache.

In [ ]:
df = pd.read_parquet(DATA / "scm_train.parquet")
print(f"Loaded Phase 1 data: {len(df)} rows, {df['user_id'].nunique()} users, {df['impression_id'].nunique()} impressions")

cdi_path = ARTIFACTS / "cdi_cache.pkl"
with open(cdi_path, "rb") as f:
    cdi_cache = pickle.load(f)
print(f"Loaded CDI cache: {len(cdi_cache)} entries")

## 3. Build Sessions
Create session objects from the SCM data, one per impression. Each session holds the user's history embedding, candidate items, and click labels.

In [ ]:
sessions = []
for imp_id, group in df.groupby("impression_id", sort=False):
    user_id = group["user_id"].iloc[0]
    emb_str = group["U_history_emb_full"].iloc[0]
    history_emb = np.array(
        emb_str if isinstance(emb_str, (list, np.ndarray))
        else eval(emb_str), dtype=np.float32
    )
    item_ids = group["item_id"].tolist()
    clicks = group["Y_click"].tolist()
    title_embs = group["I_title_emb_full"].apply(
        lambda x: np.array(
            x if isinstance(x, (list, np.ndarray)) else eval(x), dtype=np.float32
        )
    ).tolist()
    clicked_items = set(group[group["Y_click"] == 1]["item_id"].tolist())
    candidates = [
        type("C", (), {"item_id": iid, "title_emb": emb})()
        for iid, emb in zip(item_ids, title_embs)
    ]
    session = type("Session", (), {
        "user_id": user_id,
        "initial_history_emb": history_emb,
        "candidates": [candidates],
        "clicks": [clicks],
        "candidate_pool": item_ids,
        "clicked_items": clicked_items,
    })()
    sessions.append(session)

print(f"Built {len(sessions)} sessions")

## 4. Filter Sessions with CDI Coverage
Keep only sessions whose (user, candidate) pairs exist in the CDI cache.

In [ ]:
cdi_user_items = set(cdi_cache.keys())
sessions = [s for s in sessions if
            any((s.user_id, iid) in cdi_user_items for iid in s.candidate_pool)]
print(f"Sessions with CDI coverage: {len(sessions)}")

## 5. Train PPO Agent
Train a PPO agent that recommends K items per session, shaped by a combined reward: `R = w * R_click + (1-w) * CDI`.

In [ ]:
total_timesteps = 200000
n_envs = min(2, len(sessions))
K = 10
T = 1

print(f"Starting PPO training: {total_timesteps} timesteps, {n_envs} envs, K={K}, T={T}, {len(sessions)} sessions")

t0 = time.time()
model, save_path = train_ppo(
    sessions, df, cdi_cache,
    total_timesteps=total_timesteps,
    n_envs=n_envs,
    w=0.3,
    K=K,
    T=T,
    learning_rate=3e-4,
    gamma=0.95,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    n_steps=512,
    batch_size=64,
    n_epochs=10,
    tensorboard_log=str(ARTIFACTS / "tb_logs"),
    checkpoint_dir=str(ARTIFACTS / "checkpoints"),
    net_arch=[512, 256],
    verbose=0,
)
t_train = time.time() - t0
print(f"PPO training completed in {t_train:.1f} s")
print(f"Model saved to: {save_path}")

## 6. Summary
Phase 4 complete. Artifacts generated:

In [ ]:
print(f"PPO model:     {save_path}")
print(f"Checkpoints:   {ARTIFACTS / 'checkpoints'}")
print(f"TensorBoard:   {ARTIFACTS / 'tb_logs'}")
print("\nPhase 4 complete.")